# 1 - Open CV - % Gordura

### 1.1- Configurações Iniciais

In [ ]:
!pip install opencv-python
import os
import requests
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from google.colab import drive  # importa google drive
path = "/content/drive/MyDrive/Clube de Programacao-2026/Apresentacao/"
path_imagens = f"{path}imagens/"
path_saida_opencv = f"{path}open_cv/"

In [ ]:
drive.mount('/content/drive', True)  # conecta drive

Mounted at /content/drive


### 1.2 - Função para Processar imagens

In [ ]:
def processar_imagem(imagem, indice):
    # Imagem de exemplo (troque por uma foto de carne)
    imagem = cv2.imread(imagem)

    # Converter para HSV
    hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)

    # Detectar fundo branco
    mask_fundo = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 40, 255]))

    # Inverter para obter apenas a peça
    mask_peca = cv2.bitwise_not(mask_fundo)

    # Detectar gordura - Faixa para regiões claras/esbranquiçadas (gordura)
    mask_gordura = cv2.inRange(hsv, np.array([0, 0, 150]), np.array([180, 80, 255]))

    # Manter gordura apenas dentro da peça
    mask_gordura = cv2.bitwise_and(mask_gordura, mask_peca)

    # Remover ruídos
    kernel = np.ones((5, 5), np.uint8)
    mask_gordura = cv2.morphologyEx(mask_gordura, cv2.MORPH_OPEN, kernel)
    mask_gordura = cv2.morphologyEx(mask_gordura, cv2.MORPH_CLOSE, kernel)

    # Considerar apenas os pixels da peça
    pixels_totais = cv2.countNonZero(mask_peca)

    # Pixels classificados como gordura
    pixels_gordura = cv2.countNonZero(mask_gordura)

    # Percentual
    # Evita divisão por zero
    if pixels_totais == 0:
        percentual = 0
    else:
        percentual = (pixels_gordura / pixels_totais) * 100

    print(f"Percentual estimado de gordura: {percentual:.2f}%")

    # Destacar gordura em vermelho
    resultado = imagem.copy()
    resultado[mask_gordura > 0] = [0, 0, 255]

    # print em tela
    fator = 0.5
    print("Imagem original:")
    cv2_imshow(cv2.resize(imagem, None, fx=fator, fy=fator))

    print("OpenCV - Gordura detectada:")
    cv2_imshow(cv2.resize(mask_gordura, None, fx=fator, fy=fator))

    print("Resultado final:")
    cv2_imshow(cv2.resize(resultado, None, fx=fator, fy=fator))

    cv2.imwrite(f"{path_saida_opencv}{indice:02d}_original.png", imagem)
    cv2.imwrite(f"{path_saida_opencv}{indice:02d}_gordura.png", mask_gordura)
    cv2.imwrite(f"{path_saida_opencv}{indice:02d}_resultado.png", resultado)

### 1.3 - Testes

In [ ]:
# TESTANDO ------------------------------
# total de 7 imagens
for i in range(1, 8):
    imagem = f"{path_imagens}{i:02d}.png"
    print("========================")
    print(f"Imagem {i:02d}")
    print("========================")
    processar_imagem(imagem, i)

Output hidden; open in https://colab.research.google.com to view.